In [1]:
# %% [Cell 1] Imports
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF C++ logs (INFO, WARNING, ERROR)
import warnings
warnings.filterwarnings('ignore')  # Suppress Python warnings

import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Suppress TF Python-level logs
from tensorflow.keras.layers import Conv2D, Conv3D, Flatten, Dense, Reshape, BatchNormalization
from tensorflow.keras.layers import Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
from scipy.io import loadmat
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt
import os
import gc
import pandas as pd
from pathlib import Path



E0000 00:00:1771518853.250275     145 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771518853.362421     145 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771518854.273385     145 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771518854.273442     145 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771518854.273445     145 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771518854.273447     145 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
import shutil
from pathlib import Path

# Move previous results into the working directory so the script knows to skip them!
old_results = Path('/kaggle/input/your-previous-dataset-name/CNN_Results')
working_results = Path('/kaggle/working/CNN_Results')

if old_results.exists():
    shutil.copytree(old_results, working_results, dirs_exist_ok=True)
    print("Previous checkpoints restored!")

In [2]:
# %% [Cell 2] Load Data
DATASET_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_WST_Pipeline/dataset")

# Auto-detect Kaggle Environment
if Path('/kaggle/input').exists():
    print("Detected Kaggle Environment. Adjusting paths...")
    found_files = list(Path('/kaggle/input').rglob('Salinas_corrected.mat'))
    if found_files:
        DATASET_DIR = found_files[0].parent
        print(f"Found dataset at: {DATASET_DIR}")
    else:
        print("Warning: Could not find Salinas_corrected.mat in /kaggle/input")

print(f"Loading data from {DATASET_DIR}...")

data_path = DATASET_DIR / 'Salinas_corrected.mat'
gt_path = DATASET_DIR / 'Salinas_gt.mat'

if not data_path.exists():
    raise FileNotFoundError(f"File not found: {data_path}")

X_raw = loadmat(str(data_path))['salinas_corrected']
y_raw = loadmat(str(gt_path))['salinas_gt']

print(f"Data Loaded. Shape: {X_raw.shape}")



Detected Kaggle Environment. Adjusting paths...
Found dataset at: /kaggle/input/datasets/hosseinkaka1997/salinas
Loading data from /kaggle/input/datasets/hosseinkaka1997/salinas...
Data Loaded. Shape: (512, 217, 204)


In [3]:
# %% [Cell 3] Constants
test_ratio = 0.8
DEFAULT_PIXELSIZE = 25

In [4]:
# %% [Cell 4] Preprocessing Functions
def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX

def select_wavelengths(X, selected_indices):
    print(f"Selecting {len(selected_indices)} bands...")
    return X[:, :, selected_indices]

def apply_snv(X):
    print("Applying SNV...")
    mean_vec = np.mean(X, axis=2, keepdims=True)
    std_vec = np.std(X, axis=2, keepdims=True)
    return (X - mean_vec) / (std_vec + 1e-8)

def apply_msc(X, reference=None):
    print("Applying MSC...")
    original_shape = X.shape
    X_flat = X.reshape(-1, original_shape[2])
    if reference is None:
        reference = np.mean(X_flat, axis=0)
    n_samples = X_flat.shape[0]
    X_msc = np.zeros_like(X_flat)
    for i in range(n_samples):
        fit = np.polyfit(reference, X_flat[i, :], 1, full=True)
        slope = fit[0][0]
        intercept = fit[0][1]
        X_msc[i, :] = (X_flat[i, :] - intercept) / slope
    return X_msc.reshape(original_shape)

def apply_sg(X, deriv=0, window_length=30, polyorder=2):
    print(f"Applying SG (w={window_length}, p={polyorder}, d={deriv})...")
    return savgol_filter(X, window_length=window_length, polyorder=polyorder, deriv=deriv, axis=2)

def preprocess_pipeline(X, pipeline_name):
    print(f"Preprocessing Pipeline: {pipeline_name}")
    X_proc = X.copy()
    if 'SG' in pipeline_name:
        deriv = 1 if 'SG1' in pipeline_name else 0 
        X_proc = apply_sg(X_proc, deriv=deriv, window_length=30, polyorder=2)
    if 'MSC' in pipeline_name:
        X_proc = apply_msc(X_proc)
    elif 'SNV' in pipeline_name or 'SVN' in pipeline_name:
        X_proc = apply_snv(X_proc)
    return X_proc

def createImageCubes(X, y, windowSize=5, removeZeroLabels=True):
    margin = int((windowSize - 1) / 2)
    zeroPaddedX = padWithZeros(X, margin=margin)
    if removeZeroLabels:
        r_indices, c_indices = np.nonzero(y)
        num_samples = len(r_indices)
    else:
        r_indices, c_indices = np.where(y >= 0)
        num_samples = len(r_indices)
    print(f"Generating {num_samples} patches (Optimized)...")
    patchesData = np.zeros((num_samples, windowSize, windowSize, X.shape[2]), dtype=np.float32)
    patchesLabels = np.zeros((num_samples))
    for i in range(num_samples):
        r = r_indices[i]
        c = c_indices[i]
        r_pad = r + margin
        c_pad = c + margin
        patch = zeroPaddedX[r_pad - margin:r_pad + margin + 1, c_pad - margin:c_pad + margin + 1]
        patchesData[i, :, :, :] = patch
        patchesLabels[i] = y[r, c]
    if removeZeroLabels:
        patchesLabels -= 1
    return patchesData, patchesLabels



In [5]:
# %% [Cell 5] Print Info
print(f"Original X shape: {X_raw.shape}, y shape: {y_raw.shape}")

Original X shape: (512, 217, 204), y shape: (512, 217)


In [6]:
# %% [Cell 6] Helper Functions
def get_indices_from_values(selected_values, all_wavelengths):
    indices = []
    for val in selected_values:
        idx = (np.abs(all_wavelengths - val)).argmin()
        indices.append(idx)
    return sorted(list(set(indices)))

# Load Reference Wavelengths
WAVELENGTH_FILE = r"C:\Users\hosse\Desktop\Thesis_Phase1_completed\CCARS_Tuned_Hyperparameters\wavelengths_salinas_corrected_204.csv"

if Path('/kaggle/input').exists():
    found_wl = list(Path('/kaggle/input').rglob('wavelengths_salinas_corrected_204.csv'))
    if found_wl:
        WAVELENGTH_FILE = str(found_wl[0])
        print(f"Found Reference Wavelengths at: {WAVELENGTH_FILE}")

if os.path.exists(WAVELENGTH_FILE):
    all_wl = pd.read_csv(WAVELENGTH_FILE).iloc[:, 0].values
else:
    all_wl = np.arange(204)



Found Reference Wavelengths at: /kaggle/input/datasets/hosseinkaka1997/salinas/wavelengths_salinas_corrected_204.csv


In [8]:
# %% [Cell 7] Train and Evaluate Function
def train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw):
    print(f"\n{'='*50}")
    print(f"Starting Experiment: {method_name} ({pipeline_name})")
    print(f"{'='*50}")

    # --- 1. SET UP KAGGLE-SAFE DIRECTORY & CHECKPOINTING ---
    if Path('/kaggle/working').exists():
        RESULTS_DIR = Path('/kaggle/working/CNN_Results')
    else:
        RESULTS_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/CNN_Results")
    
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    safe_method_name = f"{method_name}_{pipeline_name}".replace(" ", "_").replace("/", "-")
    
    # Check if this experiment has already finished in a previous run
    report_path = RESULTS_DIR / f"classification_report_{safe_method_name}.txt"
    if report_path.exists():
        print(f"✅ Experiment {safe_method_name} is already complete! Found existing results. Skipping to save time...")
        return # Exit the function early and move to the next experiment
    # -------------------------------------------------------

    # Apply Preprocessing
    if pipeline_name != 'RAW':
        X_processed = preprocess_pipeline(X_raw, pipeline_name)
    else:
        X_processed = X_raw.copy()

    # Select Wavelengths
    X = select_wavelengths(X_processed, selected_indices)
    del X_processed
    y = y_raw.copy()

    # Adaptive pixelsize based on band count (to fit in Kaggle RAM)
    n_bands = len(selected_indices)
    if n_bands > 80:
        pixelsize = 13
    elif n_bands > 40:
        pixelsize = 19
    else:
        pixelsize = DEFAULT_PIXELSIZE
    print(f"Using pixelsize={pixelsize} for {n_bands} bands")

    # Determine num_classes
    unique_classes = np.unique(y_raw)
    unique_classes = unique_classes[unique_classes != 0]
    num_classes = len(unique_classes)
    print(f"Detected {num_classes} classes for spatial split.")

    # Block-Based Spatial Train/Test Split
    y_train_map = np.zeros_like(y_raw)
    y_test_map = np.zeros_like(y_raw)
    
    H_img, W_img = y_raw.shape
    block_size = 50  
    
    for c in range(1, num_classes + 1):
        class_indices = np.argwhere(y_raw == c)
        if len(class_indices) == 0: continue
        class_blocks_set = set()
        for r, col in class_indices:
            class_blocks_set.add((r // block_size, col // block_size))
        class_blocks = list(class_blocks_set)
        np.random.shuffle(class_blocks)
        n_test = int(len(class_blocks) * test_ratio)
        if n_test == len(class_blocks) and len(class_blocks) > 1:
            n_test -= 1
        test_blocks_set = set(class_blocks[:n_test])
        for r, col in class_indices:
            block_id = (r // block_size, col // block_size)
            if block_id in test_blocks_set:
                y_test_map[r, col] = c
            else:
                y_train_map[r, col] = c

    # Generate cubes
    Xtrain, ytrain = createImageCubes(X, y_train_map, windowSize=pixelsize, removeZeroLabels=True)
    Xtest, ytest = createImageCubes(X, y_test_map, windowSize=pixelsize, removeZeroLabels=True)
    print(f"Train shapes: {Xtrain.shape}, Test shapes: {Xtest.shape}")

    # Validation Split
    Xtrain, Xvalid, ytrain, yvalid = train_test_split(Xtrain, ytrain, test_size=0.3333, random_state=42, stratify=ytrain)

    # StandardScaler (Fit on Train)
    print("Normalizing data (StandardScaler) - Fit on Train...")
    N_tr, H, W, C = Xtrain.shape
    scaler = StandardScaler()
    Xtrain = scaler.fit_transform(Xtrain.reshape(-1, C)).reshape(N_tr, H, W, C)
    N_te = Xtest.shape[0]
    Xtest = scaler.transform(Xtest.reshape(-1, C)).reshape(N_te, H, W, C)
    N_val = Xvalid.shape[0]
    Xvalid = scaler.transform(Xvalid.reshape(-1, C)).reshape(N_val, H, W, C)
    print("Normalization Complete.")

    # Verification
    current_bands = Xtrain.shape[3]
    expected_bands = len(selected_indices)
    assert current_bands == expected_bands

    # Reshape for CNN
    input_channels = Xtrain.shape[3]
    Xtrain = Xtrain.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xtest = Xtest.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xvalid = Xvalid.reshape(-1, pixelsize, pixelsize, input_channels, 1)

    # One-Hot Encoding
    ytrain = to_categorical(ytrain)
    ytest = to_categorical(ytest)
    yvalid = to_categorical(yvalid)
    num_classes = ytrain.shape[1]

    # Model Definition
    input_shape = (pixelsize, pixelsize, input_channels, 1)

    def build_adaptive_cnn(input_shape, num_classes):
        input_layer = Input(input_shape)
        depth = input_shape[2]
        k1 = 7 if depth >= 7 else (5 if depth >= 5 else 3)
        d1 = depth - (k1 - 1)
        k2 = 5 if d1 >= 5 else (3 if d1 >= 3 else 1)
        d2 = d1 - (k2 - 1)
        k3 = 3 if d2 >= 3 else 1
        x = Conv3D(filters=8, kernel_size=(3, 3, k1), activation='relu')(input_layer)
        x = Conv3D(filters=16, kernel_size=(3, 3, k2), activation='relu')(x)
        x = Conv3D(filters=32, kernel_size=(3, 3, k3), activation='relu')(x)
        s = x.shape
        x = Reshape((s[1], s[2], s[3] * s[4]))(x)
        x = Conv2D(filters=64, kernel_size=(3,3), activation='relu')(x)
        x = Flatten()(x)
        x = Dense(units=256, activation='relu')(x)
        x = Dropout(0.4)(x)
        x = Dense(units=128, activation='relu')(x)
        x = Dropout(0.4)(x)
        outputs = Dense(units=num_classes, activation='softmax')(x)
        model = Model(inputs=input_layer, outputs=outputs)
        return model

    model = build_adaptive_cnn(input_shape, num_classes)
    adam = Adam(learning_rate=0.001, decay=1e-06)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

    # --- 2. RESUME WEIGHTS IF KERNEL DIED MID-TRAINING ---
    checkpoint_path = RESULTS_DIR / f"best_model_{safe_method_name}.keras"
    if checkpoint_path.exists():
        print("Found partial training weights! Loading them before continuing...")
        model.load_weights(str(checkpoint_path))
    
    checkpoint = ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', verbose=0, save_best_only=True, mode='max')
    callbacks_list = [checkpoint]

    # Train with Data Augmentation
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    EPOCHS = 100
    n_bands = len(selected_indices)
    if n_bands > 80:
        BATCH_SIZE = 64
    elif n_bands > 40:
        BATCH_SIZE = 128
    else:
        BATCH_SIZE = 256
    
    Xtrain_aug = Xtrain.squeeze(-1)
    datagen = ImageDataGenerator(
        rotation_range=90,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='reflect'
    )
    train_gen = datagen.flow(Xtrain_aug, ytrain, batch_size=BATCH_SIZE)
    
    def expand_dims_generator(generator):
        for x_batch, y_batch in generator:
            yield np.expand_dims(x_batch, axis=-1), y_batch

    print(f"Training {safe_method_name} with Data Augmentation for {EPOCHS} epochs...")
    history = model.fit(
        expand_dims_generator(train_gen),
        steps_per_epoch=len(Xtrain) // BATCH_SIZE,
        validation_data=(Xvalid, yvalid),
        epochs=EPOCHS, 
        callbacks=callbacks_list, 
        verbose=1
    )

    # Evaluate
    model.load_weights(str(checkpoint_path))
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    EVAL_BATCH = min(BATCH_SIZE, 64)
    loss, accuracy = model.evaluate(Xtest, ytest, batch_size=EVAL_BATCH, verbose=0)
    print(f"Test Accuracy ({safe_method_name}): {accuracy*100:.2f}%")

    # Reports
    Y_pred_test = model.predict(Xtest, batch_size=EVAL_BATCH, verbose=0)
    y_pred_test = np.argmax(Y_pred_test, axis=1)
    y_test_labels = np.argmax(ytest, axis=1)
    classification = classification_report(y_test_labels, y_pred_test)
    confusion = confusion_matrix(y_test_labels, y_pred_test)
    
    with open(report_path, 'w') as f:
        f.write(classification)
        f.write("\n\nConfusion Matrix:\n")
        f.write(str(confusion))
        f.write(f"\n\nTest Accuracy: {accuracy*100:.2f}%")

    # Plotting
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss: {safe_method_name}')
    plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy: {safe_method_name}')
    plt.legend(); plt.grid(True)
    plot_path = RESULTS_DIR / f"training_plot_{safe_method_name}.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Finished {safe_method_name}. Results saved.")

    # Memory cleanup
    del model, Xtrain, Xtest, Xvalid, ytrain, ytest, yvalid, Y_pred_test, Xtrain_aug
    tf.keras.backend.clear_session()
    gc.collect()
    print("Memory cleared.")

In [10]:
import pandas as pd
from pathlib import Path
local_wst_dir = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/WST_Results/Salinas")

pipeline_dirs = ['10_SG_MSC', '10_SG_SVN', '10_SG1_MSC', '10_SG1_SVN']
all_experiments = {}

if Path('/kaggle/input').exists():
    print("Running in Kaggle environment...")
    kaggle_base_dir = Path('/kaggle/input')
    
    for pdir in pipeline_dirs:
        search_pattern = f"*{pdir}*_selected_wavelengths.csv"
        found_csvs = list(kaggle_base_dir.rglob(search_pattern))
        
        if found_csvs:
            df = pd.read_csv(found_csvs[0])
            all_experiments[pdir] = df
            print(f"\n{pdir}: {len(df)} methods loaded")
            if 'Method' in df.columns and 'N_Selected' in df.columns:
                print(df[['Method', 'N_Selected']].to_string())
            else:
                print("Columns 'Method' and 'N_Selected' are not in this CSV.")
        else:
            print(f"Warning: No CSV found for {pdir} in Kaggle!")

else:
    print("Running in Local environment...")
    for pdir in pipeline_dirs:
        csv_file = list((local_wst_dir / pdir).glob('*_selected_wavelengths.csv'))
        if csv_file:
            df = pd.read_csv(csv_file[0])
            all_experiments[pdir] = df
            print(f"\n{pdir}: {len(df)} methods loaded")
            if 'Method' in df.columns and 'N_Selected' in df.columns:
                print(df[['Method', 'N_Selected']].to_string())
        else:
            print(f"Warning: No CSV found in {local_wst_dir / pdir}")

Running in Kaggle environment...

10_SG_MSC: 4 methods loaded
         Method  N_Selected
0          BOSS         119
1          CARS          40
2       GA-iPLS          90
3  GA-iPLS_BOSS          16

10_SG_SVN: 4 methods loaded
         Method  N_Selected
0          BOSS          32
1          CARS          26
2       GA-iPLS          80
3  GA-iPLS_BOSS          26

10_SG1_MSC: 4 methods loaded
         Method  N_Selected
0          BOSS          15
1          CARS          25
2       GA-iPLS          90
3  GA-iPLS_BOSS          56

10_SG1_SVN: 4 methods loaded
         Method  N_Selected
0          BOSS          39
1          CARS          91
2       GA-iPLS         102
3  GA-iPLS_BOSS          33


In [11]:
# %% [Cell 9] Helper: Parse WST Row
def parse_wst_row(row):
    method_name = row['Method']
    pipeline_name = row['Pipeline']
    bands_str = row['Selected_Wavelengths']
    bands_str = bands_str.replace('[', '').replace(']', '').replace('\n', '')
    selected_wavelengths = [float(x) for x in bands_str.split(',') if x.strip()]
    return selected_wavelengths, method_name, pipeline_name


In [12]:
# %% [Cell 10] Experiment 1/16: BOSS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)


Starting Experiment: BOSS (10_SG_MSC)
Preprocessing Pipeline: 10_SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 117 bands...
Using pixelsize=13 for 117 bands
Detected 16 classes for spatial split.
Generating 13034 patches (Optimized)...
Generating 41095 patches (Optimized)...
Train shapes: (13034, 13, 13, 117), Test shapes: (41095, 13, 13, 117)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
VERIFICATION: Using 117 bands. Expected: 117
Adaptive Kernels: (7), (5), (3)


I0000 00:00:1771519077.747280     145 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 13, 13, 117, 1) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 11, 11, 111, 8) │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 9, 9, 107, 16)  │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 7, 7, 105, 32)  │        13,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 3360)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 5, 5, 64)       │     1,935,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       409,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         2,064 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,400,384 (9.16 MB)

...

I0000 00:00:1771519082.122566     241 service.cc:152] XLA service 0x785e00002880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771519082.122615     241 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1771519082.668072     241 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/135 ━━━━━━━━━━━━━━━━━━━━ 19:25 9s/step - accuracy: 0.0469 - loss: 2.7661

I0000 00:00:1771519088.735699     241 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 19/135 ━━━━━━━━━━━━━━━━━━━━ 30s 261ms/step - accuracy: 0.3574 - loss: 2.0682

KeyboardInterrupt: 

In [ ]:
# %% [Cell 11] Experiment 2/16: CARS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

In [ ]:
# %% [Cell 12] Experiment 3/16: GA-iPLS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 13] Experiment 4/16: GA-iPLS_BOSS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 2: 10_SG_SVN (4 methods)
# ============================================================



In [ ]:
# %% [Cell 14] Experiment 5/16: BOSS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 15] Experiment 6/16: CARS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 16] Experiment 7/16: GA-iPLS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 17] Experiment 8/16: GA-iPLS_BOSS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 3: 10_SG1_MSC (4 methods)
# ============================================================



In [ ]:
# %% [Cell 18] Experiment 9/16: BOSS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 19] Experiment 10/16: CARS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 20] Experiment 11/16: GA-iPLS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 21] Experiment 12/16: GA-iPLS_BOSS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 4: 10_SG1_SVN (4 methods)
# ============================================================



In [ ]:
# %% [Cell 22] Experiment 13/16: BOSS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 23] Experiment 14/16: CARS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 24] Experiment 15/16: GA-iPLS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

In [ ]:
# %% [Cell 25] Experiment 16/16: GA-iPLS_BOSS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)



In [ ]:
# %% [Cell 26] Summary
print("\n" + "="*60)
print("All 16 WST experiments completed!")
print("="*60)
print("Results saved to CNN_Results/ directory.")
print("\nMethods tested: BOSS, CARS, GA-iPLS, GA-iPLS_BOSS")
print("Pipelines tested: SG_MSC, SG_SVN, SG1_MSC, SG1_SVN")
